## SOLVING PLANNING PROBLEMS


### GRAPHPLAN
<br>
The GraphPlan algorithm is a popular method of solving classical planning problems.
Before we get into the details of the algorithm, let's look at a special data structure called <b>planning graph</b>, used to give better heuristic estimates and plays a key role in the GraphPlan algorithm.

### Planning Graph
A planning graph is a directed graph organized into levels. 
Each level contains information about the current state of the knowledge base and the possible state-action links to and from that level.
The first level contains the initial state with <b>nodes representing each fluent that holds</b> in that level.
This level has <b>state-action links</b> linking each <b>state</b> to valid <b>action</b> in that state.
Each action is linked to all its <b>preconditions</b> and its <b>effect</b> states.
Based on these effects, the next level is constructed.<br><br>
The next level contains similarly structured information about the next state.
In this way, the graph is expanded using state-action links till we reach a state where all the required <b>goals</b> hold true simultaneously.
We can say that we have reached our goal if none of the goal states in the current level are mutually exclusive.
This will be explained in detail later.
<br><br>
Planning graphs only work for propositional planning problems, hence we need to eliminate all variables by generating all possible substitutions.
<br>
For example, the planning graph of the `have_cake_and_eat_cake_too` problem might look like this<br><br>
![title](images/cake_graph.jpg)
<br>
The black lines indicate links between states and actions.
<br><br>
In every planning problem, we are allowed to carry out the `no-op` action, ie, we can choose no action for a particular state.
These are called 'Persistence' actions and are represented in the graph by the small square boxes.
In technical terms, a persistence action has effects same as its preconditions.
This enables us to carry a state to the next level.
<br>
<br>
The gray lines indicate mutual exclusivity.
This means that the actions connected bya gray line cannot be taken together.
Mutual exclusivity (mutex) occurs in the following cases:
1. **Inconsistent effects**: One action negates the effect of the other. For example, _Eat(Cake)_ and the persistence of _Have(Cake)_ have inconsistent effects because they disagree on the effect _Have(Cake)_
2. **Interference**: One of the effects of an action is the negation of a precondition of the other. For example, _Eat(Cake)_ interferes with the persistence of _Have(Cake)_ by negating its precondition.
3. **Competing needs**: One of the preconditions of one action is mutually exclusive with a precondition of the other. For example, _Bake(Cake)_ and _Eat(Cake)_ are mutex because they compete on the value of the _Have(Cake)_ precondition.

In the module, planning graphs have been implemented using two classes, `Level` which stores data for a particular level and `Graph` which connects multiple levels together.
Let's look at the `Level` class.

In [4]:
import inspect
from planning import *

### Class level

Each level stores the following data
1. The current state of the level in `current_state`
2. Links from an action to its preconditions in `current_action_links`
3. Links from a state to the possible actions in that state in `current_state_links`
4. Links from each action to its effects in `next_action_links`
5. Links from each possible next state from each action in `next_state_links`. This stores the same information as the `current_action_links` of the next level.
6. Mutex links in `mutex`.
<br>
<br>
The `find_mutex` method finds the mutex links according to the points given above.
<br>
The `build` method populates the data structures storing the state and action information.
Persistence actions for each clause in the current state are also defined here. 
The newly created persistence action has the same name as its state, prefixed with a 'P'.


```text
+-------------------------------+
| Initial state (current_state) |
+-------------------------------+
             |
             | build() <> create actions (including persistence P...) 
             |           <> add current_action_links & current_state_links
             v
+-------------------------------+
| current_action_links          |
| current_state_links           |
+-------------------------------+
             |
             | compute next_action_links <> derive effects of each action
             v
+-------------------------------+
| next_action_links             |
+-------------------------------+
             |
             | populate next_state_links <> map effects -> next level states
             v
+-------------------------------+
| next_state_links (next level) |
+-------------------------------+
             |
             | find_mutex() <> detect inconsistent-effects / interference / competing-needs
             v
+-------------------------------+
| mutex (action/state pairs)    |
+-------------------------------+
             |
             | Graph/GraphPlan checks:
             |  -> Are all goals present AND non-mutex?  <> YES -> extract_solution() -> SUCCESS
             |  -> NO  -> check level-off? 
             |           <> YES -> STOP -> FAILURE (unsolvable)
             |           <> NO  -> expand_graph() -> repeat from build() on next level
             v
(repeat or terminate)


In [10]:
print(inspect.getsource(Level))

class Level:
    """
    Contains the state of the planning problem
    and exhaustive list of actions which use the
    states as pre-condition.
    """

    def __init__(self, kb):
        """Initializes variables to hold state and action details of a level"""

        self.kb = kb
        # current state
        self.current_state = kb.clauses
        # current action to state link
        self.current_action_links = {}
        # current state to action link
        self.current_state_links = {}
        # current action to next state link
        self.next_action_links = {}
        # next state to current action link
        self.next_state_links = {}
        # mutually exclusive actions
        self.mutex = []

    def __call__(self, actions, objects):
        self.build(actions, objects)
        self.find_mutex()

    def separate(self, e):
        """Separates an iterable of elements into positive and negative parts"""

        positive = []
        negative = []
        for clau

### Class Graph

The class stores a problem definition in `pddl`, 
a knowledge base in `kb`, 
a list of `Level` objects in `levels` and 
all the possible arguments found in the initial state of the problem in objects. `objects` is the set of all constant terms that appear in the initial state's ground literals.
<br><br>
The `expand_graph` method generates a new level of the graph.
This method is invoked when the goal conditions haven't been met in the current level or the actions that lead to it are mutually exclusive.
The `non_mutex_goals` method checks whether the goals in the current state are mutually exclusive.
<br>
<br>
Using the `Level` and `Graph` classes, we can define a planning graph which can either be used to provide reliable heuristics for planning problems or used in the `GraphPlan` algorithm.

```text
+----------------------+        
| Graph (graph object) |  <>--> planning_problem, Level, objects 
+----------------------+        
            | 
            | __call__() -> expand_graph()
            v
+-------------------------+        
| last_level = levels[-1] | <>-->  last_level(actions, objects)
+-------------------------+        -> Level.__call__: build -> find_mutex
            |
            | perform_actions()
            v
+----------------------+        
| new_level (next)     | <>--> appended to levels
+----------------------+


In [11]:
print(inspect.getsource(Graph))

class Graph:
    """
    Contains levels of state and actions
    Used in graph planning algorithm to extract a solution
    """

    def __init__(self, planning_problem):
        self.planning_problem = planning_problem
        self.kb = FolKB(planning_problem.initial)
        self.levels = [Level(self.kb)]
        self.objects = set(arg for clause in self.kb.clauses for arg in clause.args)

    def __call__(self):
        self.expand_graph()

    def expand_graph(self):
        """Expands the graph by a level"""

        last_level = self.levels[-1]
        last_level(self.planning_problem.actions, self.objects)
        self.levels.append(last_level.perform_actions())

    def non_mutex_goals(self, goals, index):
        """Checks whether the goals are mutually exclusive"""

        goal_perm = itertools.combinations(goals, 2)
        for g in goal_perm:
            if set(g) in self.levels[index].mutex:
                return False
        return True



### Class GraphPlan

Given a planning problem defined as a PlanningProblem, `GraphPlan` creates a planning graph stored in `graph` and expands it till it reaches a state where all its required goals are present simultaneously without mutual exclusivity.
<br><br>
Once a goal is found, `extract_solution` is called.
This method recursively finds the path to a solution given a planning graph.
In the case where `extract_solution` fails to find a solution for a set of goals as a given level, we record the `(level, goals)` pair as a **no-good**.
Whenever `extract_solution` is called again with the same level and goals, we can find the recorded no-good and immediately return failure rather than searching again. 
No-goods are also used in the termination test.
<br><br>
The `check_leveloff` method checks if the planning graph for the problem has **levelled-off**, ie, it has the same states, actions and mutex pairs as the previous level.
If the graph has already levelled off and we haven't found a solution, there is no point expanding the graph, as it won't lead to anything new.
In such a case, we can declare that the planning problem is unsolvable with the given constraints.
<br>
<br>
To summarize, the `GraphPlan` algorithm calls `expand_graph` and tests whether it has reached the goal and if the goals are non-mutex.
If so, `extract_solution` is invoked which recursively reconstructs the solution from the planning graph.
If not, then we check if our graph has levelled off and continue if it hasn't.

```text
Start
  |
  V
Initialize Graph(planning_problem) -> Graph.levels[0]
  |
  V
loop: 
  |_ graph.expand_graph()   # add one level
  |_ goal_test(last_level.kb):
    \__  solution = extract_solution(goals, -1)
    \__  RETURN solution
  |
  |-- IF check_leveloff(): RETURN None  # no solution


In [12]:
print(inspect.getsource(GraphPlan))

class GraphPlan:
    """
    Class for formulation GraphPlan algorithm
    Constructs a graph of state and action space
    Returns solution for the planning problem
    """

    def __init__(self, planning_problem):
        self.graph = Graph(planning_problem)
        self.no_goods = []
        self.solution = []

    def check_leveloff(self):
        """Checks if the graph has levelled off"""

        check = (set(self.graph.levels[-1].current_state) == set(self.graph.levels[-2].current_state))

        if check:
            return True

    def extract_solution(self, goals, index):
        """Extracts the solution"""

        level = self.graph.levels[index]
        if not self.graph.non_mutex_goals(goals, index):
            self.no_goods.append((level, goals))
            return

        level = self.graph.levels[index - 1]

        # Create all combinations of actions that satisfy the goal
        actions = []
        for goal in goals:
            actions.append(level.next_state

### Air Cargo problem
In accordance with the summary above, we have defined a helper function to carry out `GraphPlan` on the `air_cargo` problem.
The function is pretty straightforward.
Let's have a look.

In [16]:
print(inspect.getsource(air_cargo_graphPlan))
print(inspect.getsource(air_cargo))

def air_cargo_graphPlan():
    """Solves the air cargo problem using GraphPlan"""
    return GraphPlan(air_cargo()).execute()

def air_cargo():
    """
    [Figure 10.1] AIR-CARGO-PROBLEM

    An air-cargo shipment problem for delivering cargo to different locations,
    given the starting location and airplanes.

    Example:
    >>> from planning import *
    >>> ac = air_cargo()
    >>> ac.goal_test()
    False
    >>> ac.act(expr('Load(C2, P2, JFK)'))
    >>> ac.act(expr('Load(C1, P1, SFO)'))
    >>> ac.act(expr('Fly(P1, SFO, JFK)'))
    >>> ac.act(expr('Fly(P2, JFK, SFO)'))
    >>> ac.act(expr('Unload(C2, P2, SFO)'))
    >>> ac.goal_test()
    False
    >>> ac.act(expr('Unload(C1, P1, JFK)'))
    >>> ac.goal_test()
    True
    >>>
    """

    return PlanningProblem(initial='At(C1, SFO) & At(C2, JFK) & At(P1, SFO) & At(P2, JFK)',
                           goals='At(C1, JFK) & At(C2, SFO)',
                           actions=[Action('Load(c, p, a)',
                              

Let's instantiate the problem and find a solution using this helper function.

In [18]:
airCargoG = air_cargo_graphPlan()
airCargoG

[[[PCargo(C2),
   Fly(P2, JFK, SFO),
   PCargo(C1),
   PPlane(P1),
   PPlane(P2),
   PAirport(SFO),
   Load(C1, P1, SFO),
   PAirport(JFK),
   Load(C2, P2, JFK),
   Fly(P1, SFO, JFK)],
  [Unload(C2, P2, SFO), Unload(C1, P1, JFK)]]]

Each element in the solution is a valid action.
The solution is separated into lists for each level.
The actions prefixed with a 'P' are persistence actions and can be ignored.
They simply carry certain states forward.
We have another helper function `linearize` that presents the solution in a more readable format, much like a total-order planner, but it is _not_ a total-order planner.

In [21]:
linearize(airCargoG)

[Fly(P2, JFK, SFO),
 Load(C1, P1, SFO),
 Load(C2, P2, JFK),
 Fly(P1, SFO, JFK),
 Unload(C2, P2, SFO),
 Unload(C1, P1, JFK)]

Indeed, this is a correct solution.
<br>
There are similar helper functions for some other planning problems.
<br>

### Spare Tire problem.

In [20]:
spareTireG = spare_tire_graphPlan()
linearize(spareTireG)

[Remove(Flat, Axle), Remove(Spare, Trunk), PutOn(Spare, Axle)]